# Furniture Polygon Evaluation

This notebook loads polygon files whose names start with `floor_plan_variation` and computes the following metrics:

- **Outside Boundary Rate**: % of furniture partially or fully outside its room boundary.
- **Overlap Rate (OR)**: % of furniture overlapping with at least one other item.
- **Empty Room Rate (ERR)**: % of rooms with no furniture (from placement reports).
- **Placement Success Rate (PSR)**: % of expected furniture successfully placed.
- **Rule Compliance Rate (RCR)**: % of placed furniture appropriate for the room type.

If `placement_report_variation_XXX.json` files are present, ERR/PSR/RCR are computed from them.

## 1) Imports

In [ ]:
import os
import re
import json
import glob
import cv2
import numpy as np
from shapely.geometry import Polygon
from shapely.ops import unary_union
import matplotlib.pyplot as plt

print("Imports ready")

## 2) Configure folder and file loading
Update the folder path to where your PNG files live. Files must start with `floor_plan_variation`.

In [ ]:
POLYGON_FOLDER = "/Users/sanjanapatel/lora"  # <-- update if needed
PREFIX = "floor_plan_variation"

# Supported file types (png / json / npy / npz)
patterns = [
    os.path.join(POLYGON_FOLDER, f"{PREFIX}*.png"),
    os.path.join(POLYGON_FOLDER, f"{PREFIX}*.json"),
    os.path.join(POLYGON_FOLDER, f"{PREFIX}*.npy"),
    os.path.join(POLYGON_FOLDER, f"{PREFIX}*.npz"),
]

files = []
for p in patterns:
    files.extend(glob.glob(p))

files = sorted(files)
print(f"Found {len(files)} file(s)")
for f in files[:10]:
    print("-", os.path.basename(f))
if len(files) > 10:
    print(f"... and {len(files) - 10} more")

## 3) Loading polygon data
This loader accepts the following structures:
- PNG: `floor_plan_variation_XXX.png` generated by the integration notebook (extracts furniture boxes by color)
- JSON: `{ "furniture_polygons": [ [ [x,y], ... ], ... ], "room_polygon": [ [x,y], ... ] }`
- JSON: `{ "furniture": [ ... ] }` (room optional)
- NPY/NPZ: array of shape `(N, M, 2)` for furniture polygons, with optional `room_polygon` in NPZ.

In [ ]:
def _as_polygon(coords):
    if coords is None:
        return None
    return Polygon(coords)


def _extract_room_polygon_from_png(img_rgb):
    # Room mask: treat very dark pixels as walls
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    room_mask = gray > 40
    room_mask = room_mask.astype(np.uint8) * 255

    contours, _ = cv2.findContours(room_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Use largest contour as overall room boundary
    contour = max(contours, key=cv2.contourArea)
    contour = contour.squeeze()
    if contour.ndim != 2 or len(contour) < 3:
        return None

    return Polygon(contour)


def _extract_furniture_polygons_from_png(img_bgr):
    # Colors used in integration notebook (drawContours):
    # Detected = (0, 200, 0) in RGB, Rule-based = (255, 200, 0) in RGB
    # Convert to BGR color ranges
    img = img_bgr.copy()

    # Color thresholds (BGR)
    green = np.array([0, 200, 0])
    yellow = np.array([0, 200, 255])
    tol = 30

    lower_green = np.clip(green - tol, 0, 255)
    upper_green = np.clip(green + tol, 0, 255)
    lower_yellow = np.clip(yellow - tol, 0, 255)
    upper_yellow = np.clip(yellow + tol, 0, 255)

    mask_green = cv2.inRange(img, lower_green, upper_green)
    mask_yellow = cv2.inRange(img, lower_yellow, upper_yellow)
    mask = cv2.bitwise_or(mask_green, mask_yellow)

    # Morphological clean-up
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 20:
            continue
        approx = cv2.approxPolyDP(cnt, epsilon=2.0, closed=True)
        approx = approx.squeeze()
        if approx.ndim != 2 or len(approx) < 3:
            continue
        polys.append(Polygon(approx))

    return polys


def load_polygons_from_file(path):
    furniture_polys = []
    room_poly = None

    if path.endswith(".png"):
        img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if img_bgr is not None:
            furniture_polys = _extract_furniture_polygons_from_png(img_bgr)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            room_poly = _extract_room_polygon_from_png(img_rgb)

    elif path.endswith(".json"):
        with open(path, "r") as f:
            data = json.load(f)

        if "furniture_polygons" in data:
            furniture_polys = [Polygon(p) for p in data["furniture_polygons"]]
        elif "furniture" in data:
            furniture_polys = [Polygon(p) for p in data["furniture"]]
        elif isinstance(data, list):
            furniture_polys = [Polygon(p) for p in data]

        if "room_polygon" in data:
            room_poly = Polygon(data["room_polygon"])
        elif "room" in data:
            room_poly = Polygon(data["room"])

    elif path.endswith(".npy"):
        arr = np.load(path, allow_pickle=True)
        if isinstance(arr, np.ndarray):
            furniture_polys = [Polygon(p) for p in arr]

    elif path.endswith(".npz"):
        data = np.load(path, allow_pickle=True)
        if "furniture_polygons" in data:
            furniture_polys = [Polygon(p) for p in data["furniture_polygons"]]
        elif "furniture" in data:
            furniture_polys = [Polygon(p) for p in data["furniture"]]
        if "room_polygon" in data:
            room_poly = Polygon(data["room_polygon"])
        elif "room" in data:
            room_poly = Polygon(data["room"])

    return furniture_polys, room_poly


# Load all files
all_data = []
for f in files:
    furniture_polys, room_poly = load_polygons_from_file(f)
    all_data.append({
        "file": f,
        "furniture_polys": furniture_polys,
        "room_poly": room_poly,
    })

print(f"Loaded {len(all_data)} file(s)")
print("Sample:", os.path.basename(all_data[0]["file"]) if all_data else "<none>")

## 4) Metrics
We compute:
- **Outside Boundary Rate**: furniture partially or fully outside room boundaries.
- **Overlap Rate (OR)**: furniture overlapping with other items.
- **Empty Room Rate (ERR)**: rooms with no furniture (from placement reports).
- **Placement Success Rate (PSR)**: placed / expected furniture (from room rules).
- **Rule Compliance Rate (RCR)**: furniture appropriate for room type (from room rules).

In [ ]:
def parse_variation_id(filename):
    match = re.search(r"variation_(\d+)", filename)
    return int(match.group(1)) if match else None


def compute_polygon_metrics(furniture_polys, room_poly=None, clearance=5):
    total = len(furniture_polys)

    # Overlap rate (percentage of furniture overlapping with others)
    overlapping_items = set()
    overlap_areas = []
    for i in range(total):
        for j in range(i + 1, total):
            if furniture_polys[i].intersects(furniture_polys[j]):
                inter = furniture_polys[i].intersection(furniture_polys[j])
                if not inter.is_empty and inter.area > 0:
                    overlapping_items.add(i)
                    overlapping_items.add(j)
                    overlap_areas.append(inter.area)

    overlap_rate = (len(overlapping_items) / total) if total > 0 else 0.0
    overlap_area = float(np.sum(overlap_areas)) if overlap_areas else 0.0

    # Outside boundary rate (partially or fully outside room)
    outside_rate = None
    if room_poly is not None and room_poly.is_valid:
        outside_flags = [not room_poly.contains(p) for p in furniture_polys]
        outside_rate = float(np.mean(outside_flags)) if outside_flags else 0.0

    # Optional circulation metric (not required, but helpful)
    free_space_ratio = None
    if room_poly is not None and room_poly.is_valid:
        union = unary_union(furniture_polys) if furniture_polys else None
        buffered = union.buffer(clearance) if union else None
        if buffered:
            free = room_poly.difference(buffered)
            free_space_ratio = free.area / room_poly.area if room_poly.area > 0 else 0.0
        else:
            free_space_ratio = 1.0

    return {
        "total_furniture": total,
        "overlap_rate": overlap_rate,
        "overlap_area": overlap_area,
        "outside_boundary_rate": outside_rate,
        "free_space_ratio": free_space_ratio,
    }


# Compute polygon metrics per file
results = []
for item in all_data:
    metrics = compute_polygon_metrics(item["furniture_polys"], item["room_poly"], clearance=5)
    results.append({
        "file": os.path.basename(item["file"]),
        "variation_id": parse_variation_id(os.path.basename(item["file"])),
        **metrics,
    })

results[:5]

## 5) Load placement reports (for ERR, PSR, RCR)
If `placement_report_variation_XXX.json` files exist, we compute room-based metrics using the same room rules from the integration notebook.

In [ ]:
# Room rules from integration notebook
CORRECTED_FURNITURE_ROOM_MAP = {
    "living_room": ["Lsofa", "sofa", "tv", "table", "diningtable", "kitchen", "stove"],
    "bedroom_master": ["bed", "bedside", "study", "table"],
    "bedroom_guest": ["singlebed", "bedside", "study", "table"],
    "bedroom": ["singlebed", "bed", "bedside", "study", "table"],
    "bathroom": ["bathtub", "shower", "sink", "commode"],
    "toilet": ["commode", "sink"],
    "hallway": [],
    "empty_room": [],
}

MAX_FURNITURE_PER_ROOM = {
    "bed": 1,
    "singlebed": 1,
    "Lsofa": 1,
    "sofa": 1,
    "tv": 1,
    "diningtable": 1,
    "table": 1,
    "kitchen": 1,
    "stove": 1,
    "bathtub": 1,
    "shower": 1,
    "sink": 1,
    "commode": 1,
    "bedside": 2,
    "study": 1,
}


def get_room_base_type(room_name):
    room_name_lower = room_name.lower()
    compound_types = ["living_room", "bedroom_master", "bedroom_guest", "empty_room"]
    for compound in compound_types:
        if room_name_lower.startswith(compound):
            return compound
    parts = room_name_lower.split("_")
    if len(parts) >= 2 and parts[-1].isdigit():
        return "_".join(parts[:-1])
    return room_name_lower


def is_furniture_valid_for_room(furniture_name, room_type):
    valid_furniture = CORRECTED_FURNITURE_ROOM_MAP.get(room_type, [])
    name_lower = furniture_name.lower()
    for valid in valid_furniture:
        valid_lower = valid.lower()
        if name_lower == valid_lower or name_lower in valid_lower or valid_lower in name_lower:
            return True
    return False


def expected_count_for_room(room_type):
    expected = 0
    for item in CORRECTED_FURNITURE_ROOM_MAP.get(room_type, []):
        expected += MAX_FURNITURE_PER_ROOM.get(item, 1)
    return expected


def load_placement_reports(folder):
    report_files = sorted(glob.glob(os.path.join(folder, "placement_report_variation_*.json")))
    reports = {}
    for path in report_files:
        var_id = parse_variation_id(os.path.basename(path))
        if var_id is None:
            continue
        with open(path, "r") as f:
            reports[var_id] = json.load(f)
    return reports


def compute_report_metrics(report):
    rooms = report.get("rooms", {})
    total_rooms = len(rooms)
    empty_rooms = 0

    total_items = 0
    valid_items = 0
    expected_items = 0

    for room_name, room_data in rooms.items():
        count = int(room_data.get("count", 0))
        if count == 0:
            empty_rooms += 1

        room_type = get_room_base_type(room_name)
        expected_items += expected_count_for_room(room_type)

        for item in room_data.get("items", []):
            total_items += 1
            if is_furniture_valid_for_room(item.get("name", ""), room_type):
                valid_items += 1

    err = (empty_rooms / total_rooms) if total_rooms > 0 else None
    rcr = (valid_items / total_items) if total_items > 0 else None

    # PSR = placed / expected (fallback when attempted count isn't available)
    psr = (total_items / expected_items) if expected_items > 0 else None

    return {
        "empty_room_rate": err,
        "rule_compliance_rate": rcr,
        "placement_success_rate": psr,
        "rooms_total": total_rooms,
        "rooms_empty": empty_rooms,
    }


reports_by_id = load_placement_reports(POLYGON_FOLDER)
print(f"Loaded {len(reports_by_id)} placement report(s)")

In [ ]:
# Merge report-based metrics into results
report_metrics_cache = {}
for var_id, report in reports_by_id.items():
    report_metrics_cache[var_id] = compute_report_metrics(report)

for r in results:
    var_id = r.get("variation_id")
    if var_id in report_metrics_cache:
        r.update(report_metrics_cache[var_id])
    else:
        r.update({
            "empty_room_rate": None,
            "rule_compliance_rate": None,
            "placement_success_rate": None,
            "rooms_total": None,
            "rooms_empty": None,
        })

results[:5]

## 6) Summary table

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results)
summary_df

## 7) Aggregate analysis across all images
Creates a combined table (mean/median/std/min/max) across all variations and saves it for your paper.

In [ ]:
# Select numeric metrics for aggregation
metric_cols = [
    "overlap_rate",
    "outside_boundary_rate",
    "empty_room_rate",
    "placement_success_rate",
    "rule_compliance_rate",
    "free_space_ratio",
]

# Keep only columns that exist in the summary_df
metric_cols = [c for c in metric_cols if c in summary_df.columns]

agg_table = summary_df[metric_cols].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
]).round(4)

agg_table

In [ ]:
# Save tables for paper
output_dir = POLYGON_FOLDER
summary_csv = os.path.join(output_dir, "furniture_metrics_per_variation.csv")
agg_csv = os.path.join(output_dir, "furniture_metrics_aggregate.csv")

summary_df.to_csv(summary_csv, index=False)
agg_table.to_csv(agg_csv)

print(f"Saved per-variation table: {summary_csv}")
print(f"Saved aggregate table: {agg_csv}")

## 7) Visualize one example
Pick a file index to visualize room and furniture polygons.

In [ ]:
def plot_polygons(room_poly, furniture_polys, title="Example"):
    fig, ax = plt.subplots(figsize=(7, 6))
    if room_poly is not None:
        x, y = room_poly.exterior.xy
        ax.fill(x, y, alpha=0.2, fc="lightblue", ec="blue", label="Room")
    for i, p in enumerate(furniture_polys):
        x, y = p.exterior.xy
        ax.fill(x, y, alpha=0.5, label=f"F{i+1}")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.show()

example_idx = 0  # change as needed
if all_data:
    item = all_data[example_idx]
    plot_polygons(item["room_poly"], item["furniture_polys"], title=os.path.basename(item["file"]))
else:
    print("No data loaded")